# A1 paired two-class R0 query distillation

This is the single authorized M35 experiment. The frozen ResNet50 R0 teacher and MobileNetV4 A1 student independently match queries to the same GT objects; approved teacher queries add class and geometry losses while all normal GT losses remain active. The student starts from the exact saved A1 epoch-zero initialization—not epoch 140. Run top-to-bottom on a GPU runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json, os, re, shlex, shutil, subprocess, sys
MOBILE_REPO = Path('/content/mobile_adas3d')
MONODETR_REPO = Path('/content/MonoDETR')
MONODETR_COMMIT = '6994b9f512400b258c6edb75f77423beb9c126f2'
DRIVE_DATASET_ROOT = Path('/content/drive/MyDrive/datasets/kitti')
LOCAL_DATASET_ROOT = Path('/content/kitti')
SPLIT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
MONODETR_KITTI = Path('/content/monodetr_kitti_a1_distill')
R0_SELECTION = Path('/content/drive/MyDrive/mobile_adas3d_outputs/references/monodetr_r0/product_checkpoint_sweep/r0_product_selection.json')
A1_GT_ROOT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_a1_gt')
A1_GT_RUN = A1_GT_ROOT/'monodetr_a1_mnv4_vehicle_pedestrian_gt'
A1_MANIFEST = A1_GT_RUN/'experiment_manifest.json'
A1_SELECTION = A1_GT_ROOT/'product_checkpoint_sweep/a1_product_selection.json'
OUTPUT_ROOT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_a1_distill')
RUN_NAME = 'monodetr_a1_mnv4_vehicle_pedestrian_distill'
MAX_EPOCHS = 195
def run(command, cwd=None, env=None):
    command = [str(x) for x in command]; print('+', shlex.join(command), flush=True)
    merged = os.environ.copy(); merged.update(env or {})
    result = subprocess.run(command, cwd=cwd, env=merged)
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
run(['nvidia-smi'])

In [ ]:
# Rebuild the pinned, patched MonoDETR runtime.
if not MOBILE_REPO.exists(): run(['git', 'clone', 'https://github.com/Ali-RT/mobile_adas3d.git', MOBILE_REPO])
else: run(['git', 'pull', '--ff-only'], cwd=MOBILE_REPO)
if not MONODETR_REPO.exists(): run(['git', 'clone', 'https://github.com/ZrrSkywalker/MonoDETR.git', MONODETR_REPO])
run(['git', 'fetch', '--all'], cwd=MONODETR_REPO); run(['git', 'checkout', MONODETR_COMMIT], cwd=MONODETR_REPO)
run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown', 'pyyaml', 'scipy', 'opencv-python-headless', 'numba', 'scikit-image', 'tqdm', 'ninja', 'timm==1.0.20', 'pandas'])
for patch in ('patch_monodetr_colab_compat.py', 'patch_monodetr_product_taxonomy.py', 'patch_monodetr_mobilenetv4.py', 'patch_monodetr_verbose_resume.py', 'patch_monodetr_checkpoint_metadata.py', 'patch_monodetr_a1_distillation.py'):
    run([sys.executable, f'scripts/{patch}', '--monodetr-repo', MONODETR_REPO], cwd=MOBILE_REPO)
ops = MONODETR_REPO/'lib/models/monodetr/ops'; shutil.rmtree(ops/'build', ignore_errors=True)
run([sys.executable, 'setup.py', 'build', 'install'], cwd=ops, env={'MAX_JOBS': '2'})
run([sys.executable, '-c', 'import torch, timm, MultiScaleDeformableAttention; from lib.helpers.a1_distillation_loss import compute_a1_distillation_losses; print(torch.__version__, timm.__version__, torch.cuda.get_device_name(0))'], cwd=MONODETR_REPO)

In [ ]:
# Exact Chen split view; local staging is preferred, Drive aliases are accepted.
def resolve(root, names):
    for name in names:
        path = root/name
        if path.is_dir(): return path
sources = {}
for key, names in {'image_2':['training/image_2','training/image_02'], 'label_2':['training/label_2','training/label_02'], 'calib':['training/calib']}.items():
    sources[key] = resolve(LOCAL_DATASET_ROOT, names) or resolve(DRIVE_DATASET_ROOT, names)
if any(path is None for path in sources.values()): raise FileNotFoundError(f'Missing KITTI sources: {sources}')
(MONODETR_KITTI/'training').mkdir(parents=True, exist_ok=True); (MONODETR_KITTI/'ImageSets').mkdir(parents=True, exist_ok=True)
for name, target in sources.items():
    link = MONODETR_KITTI/'training'/name
    if link.is_symlink() and link.resolve() == target.resolve(): continue
    if link.exists() or link.is_symlink(): raise RuntimeError(f'Refusing to replace {link}')
    link.symlink_to(target, target_is_directory=True)
for split in ('train','val'): shutil.copy2(SPLIT_DIR/f'{split}.txt', MONODETR_KITTI/'ImageSets'/f'{split}.txt')
assert len((MONODETR_KITTI/'ImageSets/train.txt').read_text().splitlines()) == 3712
assert len((MONODETR_KITTI/'ImageSets/val.txt').read_text().splitlines()) == 3769
print('Distillation KITTI view:', MONODETR_KITTI)

In [ ]:
# Freeze paired provenance and write the distillation config.
for required in (R0_SELECTION, A1_MANIFEST, A1_SELECTION):
    if not required.is_file(): raise FileNotFoundError(required)
run([sys.executable, 'scripts/prepare_monodetr_a1_distillation.py', '--monodetr-repo', MONODETR_REPO, '--dataset-root', MONODETR_KITTI, '--r0-selection', R0_SELECTION, '--a1-manifest', A1_MANIFEST, '--a1-selection', A1_SELECTION, '--output-root', OUTPUT_ROOT, '--run-name', RUN_NAME], cwd=MOBILE_REPO)
CONFIG = MONODETR_REPO/'configs/monodetr_a1_mnv4_vehicle_pedestrian_distill.yaml'
RUN_DIR = OUTPUT_ROOT/RUN_NAME
manifest = json.loads((RUN_DIR/'experiment_manifest.json').read_text())
assert manifest['distillation_enabled'] is True and manifest['full_training_started'] is False
assert manifest['paired_gt_baseline_selection']['selected_epoch'] == 140
print(CONFIG.read_text()); print(json.dumps(manifest, indent=2))

## Required CUDA smoke gate

This performs one real augmented two-image teacher/student forward pass, GT matching, combined loss backward pass, and zero optimizer steps. Full training is blocked if no teacher pairs are approved or any value/gradient is non-finite.

In [ ]:
SMOKE_REPORT = RUN_DIR/'a1_distillation_smoke.json'
run([sys.executable, '-u', 'scripts/smoke_test_monodetr_a1_distillation.py', '--monodetr-repo', MONODETR_REPO, '--config', CONFIG, '--output', SMOKE_REPORT], cwd=MOBILE_REPO)
smoke = json.loads(SMOKE_REPORT.read_text())
assert smoke['complete'] and smoke['finite_gradients'] and smoke['approved_query_pairs'] > 0 and smoke['optimizer_steps'] == 0
print(json.dumps(smoke, indent=2))

## Real paired distillation training

The next cells resume only complete checkpoints from this exact distilled run. Training uses the same 195 epochs, batch size, optimizer, learning rate, augmentation, and validation protocol as GT-only A1. Online teacher inference will make each epoch slower.

In [ ]:
import torch, yaml
checkpoint_pattern = re.compile(r'^checkpoint_epoch_(\d+)\.pth$'); valid_checkpoints = []
for path in RUN_DIR.glob('checkpoint_epoch_*.pth'):
    match = checkpoint_pattern.match(path.name)
    if not match: continue
    try:
        payload = torch.load(path, map_location='cpu', weights_only=False); epoch = int(payload.get('epoch', -1))
        if epoch != int(match.group(1)): raise ValueError('filename/payload epoch mismatch')
        if payload.get('model_state') is None or payload.get('optimizer_state') is None: raise ValueError('state missing')
        valid_checkpoints.append((epoch, path)); print(f'Valid distilled checkpoint: epoch={epoch} size={path.stat().st_size/1e6:.1f}MB')
    except Exception as error: print(f'Skipping invalid checkpoint {path}: {type(error).__name__}: {error}')
latest = max(valid_checkpoints, default=None, key=lambda item: item[0]); run_cfg = yaml.safe_load(CONFIG.read_text())
if latest is None:
    START_EPOCH = 0; CONFIG_TO_RUN = CONFIG; print('Starting from exact frozen A1 epoch-zero initialization.')
else:
    START_EPOCH, RESUME_CHECKPOINT = latest
    run_cfg['trainer'].pop('pretrain_model', None); run_cfg['trainer']['resume_model'] = str(RESUME_CHECKPOINT); run_cfg['trainer']['max_epoch'] = MAX_EPOCHS
    CONFIG_TO_RUN = MONODETR_REPO/'configs/monodetr_a1_distill_resume.yaml'; CONFIG_TO_RUN.write_text(yaml.safe_dump(run_cfg, sort_keys=False))
    print(f'Resuming distilled A1 after epoch {START_EPOCH}: {RESUME_CHECKPOINT}')
print('Remaining epochs:', max(0, MAX_EPOCHS-START_EPOCH), 'config:', CONFIG_TO_RUN)

In [ ]:
from collections import deque
from datetime import datetime, timezone
LOG_DIR = OUTPUT_ROOT/'colab_logs'; LOG_DIR.mkdir(parents=True, exist_ok=True)
def run_training_logged(command, cwd):
    command = [str(x) for x in command]; stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S'); log_path = LOG_DIR/f'train_{RUN_NAME}_{stamp}.log'
    print('+', shlex.join(command), '\nDurable combined log:', log_path, flush=True)
    env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'; tail = deque(maxlen=120)
    with log_path.open('w', encoding='utf-8', buffering=1) as log:
        process = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout: print(line, end='', flush=True); log.write(line); tail.append(line.rstrip())
        code = process.wait()
    if code:
        print('Last captured lines:\n' + ('\n'.join(tail) if tail else '<no output>')); subprocess.run(['nvidia-smi']); subprocess.run(['df','-h','/content','/content/drive'])
        raise RuntimeError(f'Training exited {code}; durable log: {log_path}')
    return log_path
if START_EPOCH >= MAX_EPOCHS: print(f'Distilled A1 already reached epoch {START_EPOCH}; no training required.')
else:
    manifest = json.loads((RUN_DIR/'experiment_manifest.json').read_text()); manifest['full_training_started'] = True; manifest['smoke_report_sha256'] = hashlib.sha256(SMOKE_REPORT.read_bytes()).hexdigest(); (RUN_DIR/'experiment_manifest.json').write_text(json.dumps(manifest, indent=2)+'\n')
    TRAIN_LOG = run_training_logged([sys.executable, '-u', 'tools/train_val.py', '--config', CONFIG_TO_RUN], MONODETR_REPO)

## Product sweep and paired decision

Run after training. The restartable sweep applies the unchanged five gates. The final cell displays distilled metrics beside the frozen GT-only epoch-140 baseline. No compression is authorized here.

In [ ]:
SWEEP_DIR = OUTPUT_ROOT/'product_checkpoint_sweep'
run([sys.executable, '-u', 'scripts/sweep_monodetr_a1_product_checkpoints.py', '--monodetr-repo', MONODETR_REPO, '--mobile-repo', MOBILE_REPO, '--training-config', CONFIG, '--run-dir', RUN_DIR, '--dataset-root', MONODETR_KITTI, '--split-dir', SPLIT_DIR, '--output-dir', SWEEP_DIR, '--product-config', 'configs/kitti_mobileadas3d_s1.yaml', '--profile', 'colab_drive', '--score-threshold', '0.001', '--topk', '50'], cwd=MOBILE_REPO)
distilled = json.loads((SWEEP_DIR/'a1_product_selection.json').read_text()); baseline = json.loads(A1_SELECTION.read_text())
comparison = {'gt_only_epoch140': baseline['metrics'], 'distilled_selected': distilled['metrics'], 'distilled_minus_gt': {key: distilled['metrics'][key]-baseline['metrics'][key] for key in baseline['metrics']}, 'distilled_gates': distilled['gate_results'], 'all_accuracy_gates_passed': distilled['all_accuracy_gates_passed']}
COMPARISON_PATH = SWEEP_DIR/'a1_distillation_vs_gt_comparison.json'; COMPARISON_PATH.write_text(json.dumps(comparison, indent=2)+'\n')
print(json.dumps(comparison, indent=2)); print('Return:', RUN_DIR/'experiment_manifest.json', SWEEP_DIR/'a1_product_selection.json', COMPARISON_PATH)